# Loan Default Risk Analysis
## Notebook 05 — Final Load Preparation: KPIs, Risk Segmentation & Tableau-Ready Export

**Objective:**
Compute portfolio-level KPIs, build a data-driven risk segmentation framework based on
statistical findings from Notebooks 03–04, and export a Tableau-ready dataset.

**Problem Statement:**
Financial institutions face significant losses due to loan defaults and inefficient borrower
screening. This notebook operationalises findings into actionable KPIs and risk tiers to
support more effective lending decisions and improved portfolio performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load cleaned dataset
df = pd.read_csv('../data/processed/loan_final_cleaned.csv')

# Fill missing gender values — 3,773 NaN entries cause empty boxes in Tableau
df['gender'] = df['gender'].fillna('NA')
print(f'Gender values after cleanup: {df["gender"].value_counts().to_dict()}')

print(f'\nDataset shape: {df.shape}')
df.head()

---
## Section 1: Portfolio-Level KPI Computation

### KPI 1 — Portfolio Default Rate
**Definition:** Percentage of loans that defaulted out of total loans.  
**Formula:** `(Number of Defaults / Total Loans) × 100`  
**Business Use:** Baseline metric for overall portfolio health.

In [ ]:
total_loans = len(df)
total_defaults = df['status'].sum()
default_rate = (total_defaults / total_loans) * 100

print(f'Total Loans: {total_loans:,}')
print(f'Total Defaults: {total_defaults:,}')
print(f'Portfolio Default Rate: {default_rate:.2f}%')

### KPI 2 — Exposure at Default (EAD)
**Definition:** Total loan amount outstanding for defaulted borrowers.  
**Formula:** `SUM(loan_amount) WHERE status = 1`  
**Business Use:** Quantifies maximum financial exposure from defaults.

In [ ]:
total_exposure = df['loan_amount'].sum()
ead = df[df['status'] == 1]['loan_amount'].sum()
ead_pct = (ead / total_exposure) * 100

print(f'Total Portfolio Exposure: ₹{total_exposure:,.0f}')
print(f'Exposure at Default (EAD): ₹{ead:,.0f}')
print(f'EAD as % of Total Exposure: {ead_pct:.2f}%')

### KPI 3 — Average DTI by Default Status

In [ ]:
avg_dti = df.groupby('status')['debt_to_income_ratio'].mean().round(2)
avg_dti.index = ['No Default', 'Default']
print('Average DTI by Default Status:')
print(avg_dti)
print(f'\nDTI Gap: {avg_dti["Default"] - avg_dti["No Default"]:.2f} points')

### KPI 4 — Average LTV by Default Status

In [ ]:
avg_ltv = df.groupby('status')['ltv'].mean().round(2)
avg_ltv.index = ['No Default', 'Default']
print('Average LTV by Default Status:')
print(avg_ltv)
print(f'\nLTV Gap: {avg_ltv["Default"] - avg_ltv["No Default"]:.2f} points')

### KPI 5 — Default Rate by LTV Risk Bucket

In [ ]:
ltv_kpi = df.groupby('ltv_risk_bucket').agg(
    total_loans=('status', 'count'),
    defaults=('status', 'sum'),
    total_exposure=('loan_amount', 'sum')
).assign(
    default_rate_pct=lambda x: (x['defaults'] / x['total_loans'] * 100).round(2)
)
print('Default Rate by LTV Risk Bucket:')
ltv_kpi

### KPI 6 — Default Rate by Loan Type

In [ ]:
loan_type_kpi = df.groupby('loan_type').agg(
    total_loans=('status', 'count'),
    defaults=('status', 'sum'),
    avg_loan_amount=('loan_amount', 'mean')
).assign(
    default_rate_pct=lambda x: (x['defaults'] / x['total_loans'] * 100).round(2)
)
print('Default Rate by Loan Type:')
loan_type_kpi

### KPI 7 — Default Rate by Region

In [ ]:
region_kpi = df.groupby('region').agg(
    total_loans=('status', 'count'),
    defaults=('status', 'sum'),
    total_exposure=('loan_amount', 'sum')
).assign(
    default_rate_pct=lambda x: (x['defaults'] / x['total_loans'] * 100).round(2)
)
print('Default Rate by Region:')
region_kpi

### KPI Summary Table

In [ ]:
kpi_summary = pd.DataFrame({
    'KPI': [
        'Portfolio Default Rate',
        'Total Portfolio Exposure',
        'Exposure at Default (EAD)',
        'EAD as % of Total Exposure',
        'Average DTI (Defaulters)',
        'Average DTI (Non-Defaulters)',
        'Average LTV (Defaulters)',
        'Average LTV (Non-Defaulters)',
        'Average Income (Defaulters)',
        'Average Income (Non-Defaulters)'
    ],
    'Value': [
        f'{default_rate:.2f}%',
        f'{total_exposure:,.0f}',
        f'{ead:,.0f}',
        f'{ead_pct:.2f}%',
        f'{df[df["status"]==1]["debt_to_income_ratio"].mean():.2f}',
        f'{df[df["status"]==0]["debt_to_income_ratio"].mean():.2f}',
        f'{df[df["status"]==1]["ltv"].mean():.2f}',
        f'{df[df["status"]==0]["ltv"].mean():.2f}',
        f'{df[df["status"]==1]["income"].mean():,.2f}',
        f'{df[df["status"]==0]["income"].mean():,.2f}'
    ]
})
kpi_summary.style.set_caption('Portfolio KPI Summary').hide(axis='index')

---
## Section 2: Risk Segmentation Framework

Based on statistical findings from Notebooks 03–04:
- **Credit score** has near-zero correlation with default (r=-0.0028, p=0.735) — excluded
- **LTV** is the strongest predictor (r=0.0976, logistic coeff=0.2239)
- **DTI** is the second strongest (r=0.0929, logistic coeff=0.1952)
- **Product features** dominate categorical risk signals

**Risk Tiers:** Low (≤2), Medium (3–5), High (6–8), Very High (>8)

In [ ]:
def compute_risk_score(row, median_income):
    score = 0
    # DTI component
    if row['debt_to_income_ratio'] > 50:
        score += 4
    elif row['debt_to_income_ratio'] > 35:
        score += 2
    # LTV component
    ltv_bucket = row.get('ltv_risk_bucket', '')
    if ltv_bucket == 'Very High':
        score += 3
    elif ltv_bucket == 'High':
        score += 2
    elif ltv_bucket == 'Moderate':
        score += 1
    # Negative amortisation
    if row.get('neg_ammortization', 'no') == 'yes':
        score += 3
    # Lump sum payment (protective)
    if row.get('lump_sum_payment', 'no') == 'yes':
        score -= 2
    # Income
    if row['income'] < median_income:
        score += 1
    return max(score, 0)

median_income = df['income'].median()
df['risk_score'] = df.apply(lambda row: compute_risk_score(row, median_income), axis=1)

def assign_risk_tier(score):
    if score <= 2:
        return 'Low'
    elif score <= 5:
        return 'Medium'
    elif score <= 8:
        return 'High'
    else:
        return 'Very High'

df['risk_tier'] = df['risk_score'].apply(assign_risk_tier)

print('Risk Score Distribution:')
print(df['risk_score'].describe().round(2))
print(f'\nRisk Tier Distribution:')
print(df['risk_tier'].value_counts())

### Validate Risk Segmentation

In [ ]:
risk_validation = df.groupby('risk_tier').agg(
    total_loans=('status', 'count'),
    defaults=('status', 'sum'),
    total_exposure=('loan_amount', 'sum'),
    avg_risk_score=('risk_score', 'mean')
).assign(
    default_rate_pct=lambda x: (x['defaults'] / x['total_loans'] * 100).round(2),
    exposure_at_default=lambda x: x.apply(
        lambda r: df[(df['risk_tier'] == r.name) &
                     (df['status'] == 1)]['loan_amount'].sum(), axis=1)
)
tier_order = ['Low', 'Medium', 'High', 'Very High']
risk_validation = risk_validation.reindex(tier_order)
risk_validation

### Risk Tier Visualisation

In [ ]:
tier_order = ['Low', 'Medium', 'High', 'Very High']
colors = ['#5fba7d', '#E0A854', '#d65f5f', '#a50026']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rv = risk_validation.reset_index()
rv['risk_tier'] = pd.Categorical(rv['risk_tier'], categories=tier_order, ordered=True)
rv = rv.sort_values('risk_tier')

axes[0].bar(rv['risk_tier'], rv['default_rate_pct'], color=colors)
axes[0].set_title('Default Rate by Risk Tier', fontsize=13)
axes[0].set_ylabel('Default Rate (%)')
for i, (_, row) in enumerate(rv.iterrows()):
    axes[0].text(i, row['default_rate_pct'] + 0.5,
                 f"{row['default_rate_pct']:.1f}%", ha='center', fontweight='bold')

axes[1].bar(rv['risk_tier'], rv['total_loans'], color=colors)
axes[1].set_title('Loan Count by Risk Tier', fontsize=13)
axes[1].set_ylabel('Number of Loans')
for i, (_, row) in enumerate(rv.iterrows()):
    axes[1].text(i, row['total_loans'] + 50,
                 f"{row['total_loans']:,}", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Section 3: Lending Decision Mapping

In [ ]:
def assign_lending_action(tier):
    actions = {
        'Low': 'Approve — Standard Terms',
        'Medium': 'Approve with Conditions',
        'High': 'Restrict — Additional Collateral Required',
        'Very High': 'Reject / Senior Review'
    }
    return actions.get(tier, 'Review Required')

df['lending_action'] = df['risk_tier'].apply(assign_lending_action)
print('Lending Action Distribution:')
print(df['lending_action'].value_counts())

---
## Section 4: Final Tableau-Ready Export

In [ ]:
new_cols = ['risk_score', 'risk_tier', 'lending_action']
print('New columns added for Tableau:')
for col in new_cols:
    print(f'  - {col}: {df[col].nunique()} unique values')

# Verify no NaN in gender
print(f'\nGender null count: {df["gender"].isna().sum()}')
print(f'Gender values: {df["gender"].value_counts().to_dict()}')
print(f'\nFinal dataset shape: {df.shape}')

In [ ]:
output_path = '../data/processed/loan_tableau_ready.csv'
df.to_csv(output_path, index=False)
print(f'Tableau-ready dataset exported to: {output_path}')
print(f'Shape: {df.shape}')

---
## Section 5: KPI Framework Summary

| KPI | Definition | Formula |
|-----|-----------|--------|
| Portfolio Default Rate | % of loans defaulted | (Defaults / Total) × 100 |
| Exposure at Default | Total amount at risk | SUM(loan_amount) WHERE status=1 |
| Avg DTI by Status | Mean DTI split | AVG(DTI) GROUP BY status |
| Avg LTV by Status | Mean LTV split | AVG(LTV) GROUP BY status |
| Default Rate by LTV Bucket | Per-bucket default rate | Defaults/Total per bucket × 100 |
| Default Rate by Loan Type | Per-product default rate | Defaults/Total per type × 100 |
| Default Rate by Region | Geographic default rate | Defaults/Total per region × 100 |
| Risk Score | Composite risk metric | Weighted DTI + LTV + product features |
| Risk Tier | Categorical risk band | Score → Low/Medium/High/Very High |

### Key Findings:
1. Credit score is NOT a valid risk predictor (p=0.735) — excluded from segmentation
2. LTV is the strongest numerical predictor (r=0.0976)
3. DTI is the second strongest (r=0.0929)
4. Product features dominate categorical risk
5. Income correlates negatively with default